# 発展：クラスタリング ― ラベルを作らずにグループを見つける

情報Ⅱ「情報とデータサイエンス」の発展。これまで（ステップ6）は「正解ラベルを当てる」
教師あり学習でした。ここでは **正解を与えず**、似ているものどうしをグループに分ける
**クラスタリング（教師なし学習）** を2つやります。

- **A：クレーターの形と大きさ** でグループ分け → 何が出る？（既にわかっていることの再現）
- **B：場所の「夜の熱のふるまい」** でグループ分け → 周りと違う場所を見つける（探索的）

クラスタリングは「グループ番号」を作るだけで、**それが何を意味するかは人が解釈します**。
`cluster()` は各グループの特徴の平均を必ず表で見せます（ブラックボックスにしない）。

In [ ]:
from moonkit import *
from moonkit_ml import cluster, CLUSTER_METHODS
import numpy as np
print('クラスタリングの方法:', CLUSTER_METHODS)

---
## A. クレーターの形と大きさでグループ分け

使う特徴：`logD`（直径の対数）・`eccentricity`（丸さ）・`ellipticity`（扁平）・
`rim_arc_fraction`（縁がどれだけ残っているか＝劣化の目安）。

`check='区分'` を付けると、できたグループが「海／陸」のラベルに沿っているかも一緒に出ます。

In [ ]:
c = load('クレーター').dropna(subset=['diam_km', 'eccentricity', 'ellipticity', 'rim_arc_fraction']).copy()
c['logD'] = np.log10(c['diam_km'])
c = near_maria(c, scale=0.8)     # '区分' 列（海/陸）を付ける

my_k = 4        # ★ここを変える：グループの数（3〜6 で試す）
d, summary = cluster(c, ['logD', 'eccentricity', 'ellipticity', 'rim_arc_fraction'],
                     k=my_k, method='kmeans', check='区分')

**読み取り（ワークシート）**：
- どのグループが「大きい」「縁が残っている（新しそう）」「縁が消えている（古そう）」「細長い」？
- `区分=海 %` と `区分=陸 %` は、どのグループも同じくらい？　それとも1つのグループが海（or 陸）に偏る？
- → 形と大きさでグループ分けすると、**単純クレーター↔複雑クレーターの遷移**と
  **劣化（古さ）の軸**が出てきます。これは月のクレーター学で**既にわかっていること**。
  そして「海か陸か」は *どこにあるか* のラベルなので、**形のグループには表れません**
  （クラスタリングが見つけるのは「性質」であって「場所のラベル」ではない）。

---
## B. 場所の「夜の熱のふるまい」でグループ分け（探索的）

`load('夜の温度')` は Diviner の**夜の最低温度**（`temp_min_K`）と、その**緯度平均からのずれ**
（`temp_min_anomaly_K`）。岩や岩塊が多い地形は夜も熱をためて冷めにくい＝周りより暖かい
＝ 異常が正。異常は緯度の影響をすでに抜いてあるので、場所ごとの「熱のクセ」を表す。

「周りと違う温度変化を示す場所はどこか？」を、ラベルなしで探します。

In [ ]:
night = load('夜の温度')
m = night[night['lat'].abs() < 70].copy()      # Diviner が信頼できる範囲
m = near_maria(m[['lat', 'lon', 'temp_min_K', 'temp_min_anomaly_K']], scale=0.8)

# temp_min_anomaly_K は「緯度平均を引いたずれ」なので、緯度の影響はもう抜いてある。
# これと『夜そのものの寒さ temp_min_K』の2つでグループ分けする。
my_k = 4        # ★ここを変える
dn, summ = cluster(m, ['temp_min_anomaly_K', 'temp_min_K'], k=my_k, method='kmeans', check='区分')

In [ ]:
# 「夜に暖かい（異常が一番大きい）」グループはどこにある？　有名なクレーターで確かめる
warm = int(summ['temp_min_anomaly_K'].idxmax())
print(f'夜に一番暖かいグループ = グループ{warm}（岩が多いと予想）')
print('その地点数:', int(summ.loc[warm, '地点数']), '/', len(dn))

for name, lat, lon in [('Tycho', -43.3, -11.4), ('Copernicus', 9.6, -20.1),
                       ('Aristarchus', 23.7, -47.4), ('静かの海', 8.5, 31.4)]:
    row = nearest(night, lat, lon)
    print(f"  {name:12s} 夜の最低温度 {row['temp_min_K']:.0f}K   異常 {row['temp_min_anomaly_K']:+.1f}K")

**読み取り（ワークシート）**：
- 「夜に一番暖かい」グループ（＝異常が一番大きい）は、地図でどこに固まっている？
- Tycho（約1億年前・月で一番新しい大クレーター）の夜の異常は？　静かの海（古い）は？
- → 「夜に暖かい＝岩が多い＝表面が新しい（まだ砂に覆われていない）」。
  Tycho（異常 +35 K）や Aristarchus（+26 K）が飛び抜けます。**「周りと違うふるまいを示す場所」を、
  ラベルなしのデータから見つけた**例です（探索的な発見）。
- 注意：この異常は最も岩の多い場所（新しいクレーター）を拾いますが、
  「若いクレーター全部」を当てるものではありません（`temp_min_anomaly_K` の上位を
  DeepCraters の年代と突き合わせても、きれいには対応しません）。
- `temp_min_K` を features から外して `temp_min_anomaly_K` だけにすると？　グループはどう変わる？

---
## まとめ

| | やったこと | 出てきたもの |
|---|---|---|
| **A** | クレーターの形・大きさでグループ分け | 単純↔複雑の遷移、劣化の軸（**既にわかっていることの再現**）。海／陸では分かれない |
| **B** | 場所の夜の熱のふるまいでグループ分け | 「夜に暖かい＝岩が多い」場所（Tycho の噴出物など）。**周りと違う場所の発見** |

- クラスタリングは**正解ラベルを作りません**。「グループ番号」を出すだけで、
  それが何を意味するか（劣化？　岩の多さ？）は**人が特徴の表を見て解釈**します。
- `my_k` を変えるとグループの切れ目は動きます。「正しい k」は無く、**解釈しやすいか**で選びます。
- A のように「既に知っていることが出る」のも、方法が正しく働いている確認になります。
  B のように「予想していなかった場所」が出たら、次は別のデータで確かめます（探究の続き）。